<div style="background: linear-gradient(135deg, #1a1a2e, #16213e, #0f3460); padding: 40px; border-radius: 12px; text-align: center; color: white; margin-bottom: 20px;">
<h1 style="font-size: 2.5em; margin: 0;">Sequential Models</h1>
</div>

---

- Read the explanations carefully
- Fill in the `# YOUR CODE HERE` cells
- Answer the written questions in the markdown cells (double-click to edit)
- Run every cell — even if you didn't write it

> **Prerequisite check:** You should know what a neural network is, what forward/backward propagation does, and be comfortable with basic Python and NumPy.

---

## Roadmap

| Section | Topic | Type |
|---------|-------|------|
| 1 | Why ANNs Fail on Sequences | Reading + Q&A |
| 2 | RNNs — Concept & Architecture | Reading + Diagram |
| 3 | RNNs in PyTorch | Coding |
| 4 | The Vanishing Gradient Problem | Reading + Q&A |
| 5 | LSTMs — Architecture Deep Dive | Reading + Coding |
| 6 | GRUs — The Lighter Alternative | Reading + Coding |
| 7 | Mini Project: Next-Word Prediction | Full Coding Task |
| 8 | Reflection Questions | Written |


---
## Setup — Run This First

In [ ]:
# Run this cell first — installs and imports everything you need
# If something fails, read the error carefully before asking for help!

!pip install torch numpy matplotlib seaborn -q

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(" All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'GPU ' if torch.cuda.is_available() else 'CPU '}")

---
# Section 1 — Why ANNs Fail on Sequences

##  Read This

You already know that a standard **Feedforward ANN** takes a fixed-size input, passes it through layers, and gives an output. It works great for:
- Classifying an image (fixed size grid of pixels)
- Predicting house prices (fixed set of features)

But think about **language**. The sentence:
> *"The cat sat on the mat"*

has 6 words. But the sentence:
> *"She opened the old, dusty, creaking wooden door"*

has 8 words. ANNs expect **fixed-size inputs** — so what do you do?

Worse, even if you somehow fixed the length, **order matters** in language:
> *"Dog bites man"* ≠ *"Man bites dog"*

An ANN treats position 1, 2, 3 independently — it has no concept of *what came before*.

### The Core Problem: **No Memory**

When an ANN processes the word *"it"* in:
> *"I left my phone at home because it was dead"*

It has no idea that *"it"* refers to *"phone"* — because it doesn't remember the earlier words.

**This is exactly the problem Sequential Models solve.**

---

### Connection to DocForge

In our project, when the model reads a function like:
```python
def calculate_area(radius):
    return 3.14 * radius * radius
```
It reads `calculate`, then `_area`, then `(`, then `radius`... each token *depends on what came before* to understand the full meaning. That's why sequential processing matters.


---
# Section 2 — Recurrent Neural Networks (RNNs)

##  Read This

An **RNN** solves the memory problem by maintaining a **hidden state** — a vector that carries information from one step to the next.

### Architecture

At each time step `t`, the RNN takes:
- `x_t` — the current input (e.g., the current word/token)
- `h_{t-1}` — the hidden state from the previous step (the "memory")

And produces:
- `h_t` — the new hidden state
- `y_t` — the output at this step (optional, depends on task)

### The Core Equation

```
h_t = tanh(W_h * h_{t-1}  +  W_x * x_t  +  b)
```

Let's break this down:
- `W_x` — weight matrix for the current input
- `W_h` — weight matrix for the previous hidden state
- `b` — bias
- `tanh` — activation function (keeps values between -1 and 1)

**The key insight:** the same weights `W_h` and `W_x` are used at **every time step**. This is called **weight sharing**, and it's what allows RNNs to handle sequences of any length.

### Unrolling an RNN

```
x_1  →  [RNN cell]  →  h_1
x_2  →  [RNN cell]  →  h_2     (h_1 also fed in)
x_3  →  [RNN cell]  →  h_3     (h_2 also fed in)
x_4  →  [RNN cell]  →  h_4     (h_3 also fed in)
                         ↓
                       output
```

### Types of RNN Tasks

| Task Type | Description | Example |
|-----------|-------------|---------|
| Many-to-One | Sequence → Single output | Sentiment analysis |
| One-to-Many | Single input → Sequence | Image captioning |
| Many-to-Many | Sequence → Sequence | Machine translation |
| Many-to-Many (synced) | Each input → Each output | POS tagging |


###  Q2.1


 — Manual Calculation

**Work through this by hand (or use the code cell below).**

Given:
- Input sequence: `x_1 = [1.0]`, `x_2 = [0.5]`
- Initial hidden state: `h_0 = [0.0]`
- `W_x = [0.8]`, `W_h = [0.5]`, `bias = 0.1`
- Activation: `tanh`

**Calculate h_1 and h_2 step by step.**

> Show your working:
>
> h_1 = tanh(W_h * h_0 + W_x * x_1 + b) = tanh( __ * __ + __ * __ + __ ) = tanh( __ ) = __
>
> h_2 = tanh(W_h * h_1 + W_x * x_2 + b) = tanh( __ * __ + __ * __ + __ ) = tanh( __ ) = __

In [ ]:
# Verify your manual calculation here
import numpy as np

W_x = 0.8
W_h = 0.5
b   = 0.1
h0  = 0.0
x1  = 1.0
x2  = 0.5

# YOUR CODE HERE
# Calculate h1
h1 = None  # replace with the formula

# Calculate h2
h2 = None  # replace with the formula

print(f"h1 = {h1}")
print(f"h2 = {h2}")

---
# Section 3 — RNNs in PyTorch

## Read This

PyTorch provides a built-in `nn.RNN` module. Here's what you need to know:

```python
rnn = nn.RNN(
    input_size  = 10,   # size of each input vector x_t
    hidden_size = 20,   # size of the hidden state h_t
    num_layers  = 1,    # how many RNN layers stacked
    batch_first = True  # input shape: (batch, seq_len, input_size)
)
```

When you call `rnn(input, h0)`:
- `input` shape: `(batch_size, sequence_length, input_size)`
- `h0` shape: `(num_layers, batch_size, hidden_size)` — initial hidden state
- Returns: `output` (all hidden states), `h_n` (final hidden state)

### A Note on Shapes
Shapes are the **#1 source of bugs** when working with sequence models. Always print and check shapes!


In [ ]:
# ── DEMO: Basic RNN in PyTorch ──────────────────────────────────────────
# Run this cell to see how nn.RNN works. Read the output carefully.

rnn = nn.RNN(input_size=5, hidden_size=8, num_layers=1, batch_first=True)

# Fake input: batch_size=2, sequence_length=6, input_size=5
x  = torch.randn(2, 6, 5)
h0 = torch.zeros(1, 2, 8)   # (num_layers, batch_size, hidden_size)

output, h_n = rnn(x, h0)

print("Input shape:         ", x.shape)
print("Initial hidden shape:", h0.shape)
print("Output shape:        ", output.shape, "  ← hidden state at every time step")
print("Final hidden shape:  ", h_n.shape,   "  ← hidden state at last time step only")
print()
print("output[:, -1, :] == h_n[0]:", torch.allclose(output[:, -1, :], h_n[0]))

In [ ]:
# ── EXERCISE 3.1: Build a Simple RNN Classifier ──────────────────────────
#
# Task: Build a class called SimpleRNNClassifier that:
#   - Takes sequences of input_size=10
#   - Has a hidden_size of 16
#   - Has a fully connected layer that maps hidden state → 2 classes
#   - Uses the FINAL hidden state (h_n) to make the classification
#
# This is a Many-to-One architecture (whole sequence → one label)

class SimpleRNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleRNNClassifier, self).__init__()
        # YOUR CODE HERE
        # Define self.rnn and self.fc
        pass

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        # YOUR CODE HERE
        # 1. Pass x through self.rnn (initialize h0 as zeros)
        # 2. Take the final hidden state
        # 3. Pass through self.fc
        # 4. Return output
        pass


# Test your model
model = SimpleRNNClassifier(input_size=10, hidden_size=16, num_classes=2)
test_input = torch.randn(4, 7, 10)  # batch=4, seq_len=7, input=10
output = model(test_input)
print("Output shape:", output.shape)  # should be (4, 2)
print(" Correct!" if output.shape == (4, 2) else " Check your code")

---
# Section 4 — The Vanishing Gradient Problem

##  Read This

RNNs sound great in theory. In practice, they struggle badly with **long sequences**. Here's why.

### How Backpropagation Works in RNNs

Training an RNN uses **Backpropagation Through Time (BPTT)**. To update the weights, we compute gradients going backward through each time step. For a sequence of length T, the gradient at step 1 involves:

```
∂Loss/∂W  ∝  W_h^T  (W_h multiplied T times)
```

### What Goes Wrong

If `|W_h| < 1`: multiply it by itself 50 times → number gets **extremely close to zero**  
→ The gradient **vanishes** — early time steps get no training signal

If `|W_h| > 1`: multiply it by itself 50 times → number **explodes to infinity**  
→ The gradient **explodes** — training becomes unstable

### What This Means in Practice

Consider this sentence:
> *"The trophy didn't fit in the suitcase because __[it]__ was too big"*

To know what *"it"* refers to, the model needs to remember words from many steps back. With vanishing gradients, a basic RNN simply **forgets**.

For DocForge, this means a basic RNN reading a long function would forget the function name by the time it reaches the return statement!

### Visualizing the Vanishing Gradient

In [ ]:
# DEMO: Visualize how gradients vanish over time steps

steps = np.arange(1, 51)

w_vanish  = 0.9  # |W| < 1 → gradients shrink
w_explode = 1.1  # |W| > 1 → gradients grow
w_good    = 1.0  # |W| = 1 → gradients stable (rare in practice)

grad_vanish  = w_vanish  ** steps
grad_explode = np.clip(w_explode ** steps, 0, 1e6)  # clip for display
grad_good    = w_good    ** steps

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, grad_vanish, color='#e74c3c', linewidth=2)
axes[0].set_title('Vanishing Gradient\n(W=0.9)', fontweight='bold')
axes[0].set_xlabel('Time steps back')
axes[0].set_ylabel('Gradient magnitude')
axes[0].axhline(y=0.01, color='gray', linestyle='--', alpha=0.5, label='Effectively zero')
axes[0].legend()

axes[1].plot(steps, grad_explode, color='#e67e22', linewidth=2)
axes[1].set_title('Exploding Gradient\n(W=1.1)', fontweight='bold')
axes[1].set_xlabel('Time steps back')

axes[2].plot(steps, grad_good, color='#2ecc71', linewidth=2)
axes[2].set_title('Stable Gradient\n(W=1.0)', fontweight='bold')
axes[2].set_xlabel('Time steps back')

plt.suptitle('Gradient Magnitude over Time Steps', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Section 5 — Long Short-Term Memory (LSTM)

##  Read This

LSTMs (introduced by Hochreiter & Schmidhuber, 1997) were designed **specifically** to fix the vanishing gradient problem. The key idea: give the network explicit **gates** to control what to remember and what to forget.

### Two States Instead of One

Unlike a basic RNN (which only has `h_t`), an LSTM has:
- `h_t` — **Hidden state** (short-term memory, same as RNN)
- `C_t` — **Cell state** (long-term memory, the big innovation)

The cell state is like a **conveyor belt** running through the sequence — information flows along it with only minor, controlled changes.

### The Four Gates

```
─────────────────────────────────────────────────────────────
 FORGET GATE:   f_t = σ(W_f · [h_{t-1}, x_t] + b_f)
                → "How much of the old cell state to keep?"
                → Output: 0 (forget everything) to 1 (keep everything)

 INPUT GATE:    i_t = σ(W_i · [h_{t-1}, x_t] + b_i)
                → "How much new info to write to cell state?"

 CANDIDATE:    C̃_t = tanh(W_c · [h_{t-1}, x_t] + b_c)
                → "What new info to potentially write?"

 UPDATE CELL:  C_t = f_t ⊙ C_{t-1}  +  i_t ⊙ C̃_t
                → Forget some old, add some new

 OUTPUT GATE:   o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
                → "What part of cell state to output?"

 HIDDEN STATE:  h_t = o_t ⊙ tanh(C_t)
─────────────────────────────────────────────────────────────
```

Where `σ` = sigmoid, `⊙` = element-wise multiplication.

### Intuition with an Example

Imagine reading: *"John, who studied in Paris for 5 years, **is** a doctor"*

- When we hit *"John"*, the LSTM stores it in the cell state
- The long parenthetical phrase *"who studied in Paris for 5 years"* — the forget gate keeps *"John"* alive
- When predicting the verb *"is"*, the model correctly uses *"John"* (singular) because it never forgot it

A basic RNN might have lost *"John"* by the time it reaches *"is"*.


In [ ]:
# DEMO: Visualize sigmoid vs tanh — run this and observe

x = np.linspace(-5, 5, 200)
sigmoid = 1 / (1 + np.exp(-x))
tanh    = np.tanh(x)

plt.figure(figsize=(9, 4))
plt.plot(x, sigmoid, color='#3498db', linewidth=2.5, label='Sigmoid σ (range: 0 to 1)')
plt.plot(x, tanh,    color='#e74c3c', linewidth=2.5, label='Tanh (range: -1 to 1)')
plt.axhline(0, color='gray', linewidth=0.8, linestyle='--')
plt.axhline(1, color='#3498db', linewidth=0.8, linestyle=':')
plt.axhline(-1, color='#e74c3c', linewidth=0.8, linestyle=':')
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('Sigmoid vs Tanh', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Sigmoid at 0:", round(1/(1+np.exp(0)), 3),  " → gates start at 0.5 (neutral)")
print("Tanh    at 0:", round(np.tanh(0), 3), "    → candidate values centered at 0")

In [ ]:
# ── EXERCISE 5.1: LSTM Classifier ────────────────────────────────────────
#
# Task: Build an LSTMClassifier — same structure as your RNN classifier
# from Exercise 3.2, but using nn.LSTM instead of nn.RNN.
#
# Important: nn.LSTM returns (output, (h_n, c_n)) — notice the tuple!
# You still want to use h_n (not c_n) for classification.
#
# Hint: h_n shape is (num_layers, batch_size, hidden_size)
#       To get the last layer: h_n[-1]  →  shape: (batch_size, hidden_size)

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x):
        # YOUR CODE HERE
        # Remember: LSTM returns (output, (h_n, c_n))
        pass


# Test
lstm_model = LSTMClassifier(input_size=10, hidden_size=16, num_classes=3)
test_input  = torch.randn(4, 7, 10)
output      = lstm_model(test_input)
print("Output shape:", output.shape)  # should be (4, 3)
print("✅ Correct!" if output.shape == (4, 3) else "❌ Check your code")

In [ ]:
# ── EXERCISE 5.4: Count Parameters ───────────────────────────────────────
# 
# Now compare how many parameters an RNN vs LSTM has for the same config.
# This will tell you why LSTMs are more powerful but heavier.
#
# A parameter counter is given — you just need to instantiate both models
# and call count_parameters on each.

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# YOUR CODE HERE
# Create an RNN-based classifier and an LSTM-based classifier
# with input_size=10, hidden_size=32, num_classes=2
# Then print their parameter counts

rnn_clf  = None  # replace
lstm_clf = None  # replace

print(f"RNN  parameters: {count_parameters(rnn_clf):,}")
print(f"LSTM parameters: {count_parameters(lstm_clf):,}")
print(f"Ratio (LSTM/RNN): {count_parameters(lstm_clf)/count_parameters(rnn_clf):.2f}x")

---
# Section 6 — Gated Recurrent Unit (GRU)

##  Read This

GRUs (Cho et al., 2014) are a **streamlined version of LSTMs**. They keep the gating mechanism but simplify the architecture:

| | LSTM | GRU |
|-|------|-----|
| States | h_t + C_t (two) | h_t (one only) |
| Gates | 3 (forget, input, output) | 2 (reset, update) |
| Parameters | ~4x RNN | ~3x RNN |
| Speed | Slower | Faster |
| Performance | Often slightly better on long sequences | Often competitive |

### GRU Equations

```
UPDATE GATE:  z_t = σ(W_z · [h_{t-1}, x_t])
              → "How much to update the hidden state?"
              → z_t ≈ 1: keep old state; z_t ≈ 0: use new state

RESET GATE:   r_t = σ(W_r · [h_{t-1}, x_t])
              → "How much of past hidden state to use for candidate?"

CANDIDATE:   h̃_t = tanh(W · [r_t ⊙ h_{t-1}, x_t])
              → New candidate hidden state

FINAL:        h_t = (1 - z_t) ⊙ h_{t-1}  +  z_t ⊙ h̃_t
              → Blend old and new based on update gate
```

### Key Insight: The Update Gate is Clever

Notice the final equation: `h_t = (1 - z_t) * h_{old} + z_t * h̃_new`

This is a **weighted blend** — if `z_t = 0.8`, we keep 20% old and take 80% new. The model learns how much to mix automatically!

### When to Use GRU vs LSTM?

- **GRU**: Faster training, fewer parameters, good default choice for shorter sequences
- **LSTM**: Better for very long-range dependencies, typically more accurate on complex tasks
- **In practice**: Try both, pick what works!


### ✏️ Q6.1 — Compare and Contrast

Fill in the comparison table:

| Feature | RNN | LSTM | GRU |
|---------|-----|------|-----|
| Number of gates | | | |
| Number of states (h, c) | | | |
| Handles long dependencies? | | | |
| Relative speed | | | |
| When would you choose this? | | | |

> 📝 *Fill in the table above*

In [ ]:
# ── EXERCISE 6.2: GRU Classifier ─────────────────────────────────────────
#
# Task: Build a GRUClassifier using nn.GRU.
#
# Note: nn.GRU returns (output, h_n) — same as nn.RNN, NOT like LSTM!
# (No cell state, so no tuple inside tuple)

class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(GRUClassifier, self).__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x):
        # YOUR CODE HERE
        pass


# Test
gru_model = GRUClassifier(input_size=10, hidden_size=16, num_classes=3)
output    = gru_model(test_input)
print("Output shape:", output.shape)  # should be (4, 3)
print("✅ Correct!" if output.shape == (4, 3) else "❌ Check your code")

# Compare parameter counts
gru_params  = count_parameters(gru_model)
lstm_params = count_parameters(LSTMClassifier(10, 16, 3))
rnn_params  = count_parameters(SimpleRNNClassifier(10, 16, 2))
print(f"\nParameter comparison (hidden_size=16):")
print(f"  RNN:  {rnn_params}")
print(f"  GRU:  {gru_params}")
print(f"  LSTM: {lstm_params}")

---
# Section 7 —Character-Level Language Model

## Read This

You're going to build a **character-level language model** — a model that reads a sequence of characters and predicts the next character. This is one of the classic demonstrations of how sequential models learn structure from text.

**Why character-level?** It's simpler than word-level — no vocabulary explosion, easier to train on small data, and still shows the key ideas.

**The task:** Train on a small text corpus → given a sequence of characters, predict the next character.

### What You'll Build

```
Input:  ['d', 'e', 'f', ' ', 'c', 'a', 'l']
Target: ['e', 'f', ' ', 'c', 'a', 'l', 'c']
                                          ↑ each output is the next character
```

This is a **Many-to-Many (synced)** architecture.

### The Training Pipeline

```
Text → Character vocabulary → Integer encoding → One-hot or embedding
  → LSTM → Linear layer → Softmax → Predicted next char → Cross-entropy loss
```

We'll use **Python function signatures** as the training text — fitting for DocForge!


In [ ]:
# ── STEP 1: Data Preparation ──────────────────────────────────────────────
# This cell is fully provided — read it and understand each part.

# Training text: Python function signatures (DocForge relevant!)
text = """
def calculate_area(radius):
def sort_list(items, reverse=False):
def read_file(filepath, encoding='utf-8'):
def find_max(numbers):
def merge_dicts(dict_a, dict_b):
def is_palindrome(string):
def count_words(text):
def format_output(data, indent=4):
def load_model(path, device='cpu'):
def tokenize_code(source, language='python'):
""".strip()

# Build character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Text length: {len(text)} characters")
print(f"Unique characters: {vocab_size}")
print(f"Characters: {''.join(chars)}")

# Create mappings
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Encode entire text as integers
encoded = [char_to_idx[ch] for ch in text]
print(f"\nFirst 10 encoded: {encoded[:10]}")
print(f"Decoded back: {''.join([idx_to_char[i] for i in encoded[:10]])}")

In [ ]:
# ── STEP 2: Create Training Sequences ─────────────────────────────────────
# Provided — read and understand

SEQ_LEN = 20  # length of each input sequence

def create_sequences(encoded_text, seq_len):
    """Create (input_sequence, target_sequence) pairs."""
    inputs, targets = [], []
    for i in range(len(encoded_text) - seq_len):
        inputs.append(encoded_text[i : i + seq_len])
        targets.append(encoded_text[i + 1 : i + seq_len + 1])  # shifted by 1
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

X, y = create_sequences(encoded, SEQ_LEN)
print(f"Input sequences shape:  {X.shape}")
print(f"Target sequences shape: {y.shape}")
print(f"\nExample:")
print(f"  Input:  {''.join([idx_to_char[i.item()] for i in X[0]])}")
print(f"  Target: {''.join([idx_to_char[i.item()] for i in y[0]])}")

In [ ]:
# ── STEP 3: Build the Character-Level LSTM Model ──────────────────────────
#
# YOUR TASK: Complete the CharLSTM class.
#
# Architecture:
#   1. nn.Embedding(vocab_size, embed_dim) — converts char indices to vectors
#   2. nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True)
#   3. nn.Linear(hidden_size, vocab_size) — predict next char from vocab
#
# forward() should return logits of shape (batch, seq_len, vocab_size)

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers):
        super(CharLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # YOUR CODE HERE
        # Define self.embedding, self.lstm, self.fc
        pass

    def forward(self, x):
        # x shape: (batch_size, seq_len)  ← integer indices
        # YOUR CODE HERE
        # 1. Pass through embedding → (batch, seq_len, embed_dim)
        # 2. Pass through lstm
        # 3. Pass through fc → (batch, seq_len, vocab_size)
        # 4. Return the logits
        pass


# Instantiate
EMBED_DIM   = 32
HIDDEN_SIZE = 128
NUM_LAYERS  = 2

char_model = CharLSTM(vocab_size, EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS)
print(f"Model parameters: {count_parameters(char_model):,}")

# Quick shape check
test_out = char_model(X[:4])
print(f"Output shape: {test_out.shape}")  # should be (4, SEQ_LEN, vocab_size)
print("✅ Correct!" if test_out.shape == (4, SEQ_LEN, vocab_size) else "❌ Check your code")

In [ ]:
# ── STEP 4: Training Loop ─────────────────────────────────────────────────
# Provided — read and understand, then run.

from torch.utils.data import DataLoader, TensorDataset

EPOCHS     = 80
BATCH_SIZE = 32
LR         = 0.005

dataset    = TensorDataset(X, y)
loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(char_model.parameters(), lr=LR)

loss_history = []

for epoch in range(EPOCHS):
    char_model.train()
    total_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = char_model(xb)            # (batch, seq_len, vocab_size)
        # Reshape for CrossEntropyLoss: (batch*seq_len, vocab_size) vs (batch*seq_len,)
        loss   = criterion(logits.reshape(-1, vocab_size), yb.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(char_model.parameters(), 1.0)  # gradient clipping!
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    loss_history.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}]  Loss: {avg_loss:.4f}")

# Plot training curve
plt.figure(figsize=(8, 3))
plt.plot(loss_history, color='#3498db', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss — Character-Level LSTM', fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── STEP 5: Generate Text ──────────────────────────────────────────────────
# Provided — run after training

def generate_text(model, seed_text, length=80, temperature=0.8):
    """
    Generate text starting from seed_text.
    temperature: lower = more deterministic, higher = more random
    """
    model.eval()
    result = list(seed_text)
    input_seq = [char_to_idx.get(ch, 0) for ch in seed_text[-SEQ_LEN:]]

    with torch.no_grad():
        for _ in range(length):
            x_in  = torch.tensor([input_seq[-SEQ_LEN:]], dtype=torch.long)
            logits = model(x_in)[0, -1, :]  # last time step
            probs  = torch.softmax(logits / temperature, dim=0).numpy()
            next_idx = np.random.choice(len(probs), p=probs)
            result.append(idx_to_char[next_idx])
            input_seq.append(next_idx)

    return ''.join(result)

print("Generated text (seed='def '):")
print("-" * 50)
print(generate_text(char_model, "def ", length=100))
print("-" * 50)
print("\nGenerated text (seed='def load'):")
print("-" * 50)
print(generate_text(char_model, "def load", length=80))

### ✏️ Q7.1 — Analysis

1. Look at the generated text. Does it look Python-like? What patterns did the model learn?
2. In the training loop, there's a line: `torch.nn.utils.clip_grad_norm_(char_model.parameters(), 1.0)`. What does this do and why is it there?
3. What does the `temperature` parameter control in text generation? What happens if you set it to 0.1 vs 2.0? *(Try it!)*
4. This model was only trained on 10 function signatures. If we trained it on thousands of Python functions (like the Code2Doc dataset used in DocForge), what do you think would improve?

> 📝 *Your answers:*
> 
> 1. 
> 2. 
> 3. 
> 4. 

---
# Section 8 — Bonus Challenge 🔥

*(Optional — but highly recommended if you want to go deeper)*

### Challenge 1: Stacked LSTM
Modify your `CharLSTM` to use `num_layers=3` and add **dropout between layers** (`dropout=0.3` parameter in nn.LSTM). Retrain and compare the loss curves. Does more depth help?

### Challenge 2: GRU vs LSTM Race
Build a `CharGRU` (same architecture as `CharLSTM` but using GRU). Train both for 80 epochs. Plot their loss curves on the same graph. Which converges faster? Which gets lower final loss?

### Challenge 3: Bidirectional RNN
Look up `nn.LSTM(bidirectional=True)`. What does bidirectionality mean? Why is it useful for *understanding* text but problematic for *generating* text? Build a bidirectional LSTM classifier and explain the output shape change.


In [ ]:
# YOUR BONUS CODE HERE


---
# Section 9 — Final Reflection

*Answer these after completing the worksheet. Write at least 2-3 sentences each.*

### ✏️ R1 — Big Picture
In the DocForge project, Phase 5 covers Transformer models (CodeT5, CodeBERT). Now that you understand RNNs and LSTMs, what do you think Transformers might do *differently* to handle sequences? What limitation of LSTMs might they fix?

> 📝 *Your answer:*
> 
> ...

---

### ✏️ R2 — Honest Self-Assessment
What part of this worksheet was most confusing? What would you look up or experiment with more before moving to Transformers?

> 📝 *Your answer:*
> 
> ...

---

### ✏️ R3 — Application
If you had to use an LSTM to help with one specific part of the DocForge pipeline (parsing code, generating docstrings, evaluating output quality), which would you pick and how would you set it up?

> 📝 *Your answer:*
> 
> ...

---